# Check 1: Are village roads and hospitals mapped?

**Why this matters:** the whole "cut off" analysis depends on OpenStreetMap (OSM) having the small village roads. If only highways are mapped, every village will look cut off and the results mean nothing. This is the biggest risk in the project, so we check it first.

**What this notebook does**
1. Downloads one OSM file for northeast India from Geofabrik (about 100 MB, updated daily). We read everything from this file, so we never depend on the busy public OSM servers.
2. Cuts out each district and measures how much of its road network is village roads versus highways.
3. Counts health facilities.
4. Shows a satellite map with OSM roads on top, so you can see with your own eyes whether village roads are missing.

**Run it in Google Colab.** It takes about 5 to 10 minutes.

Version: 2026-10-26 (local OSM file)

In [1]:
!pip install -q osmnx folium
!apt-get -qq install -y osmium-tool > /dev/null
!osmium --version | head -1

osmium version 1.16.0


In [2]:
# Get the project code (config, preprocessing, metrics) from GitHub.
import os, sys, subprocess
REPO_URL = "https://github.com/Matrixco123/cutoff-flood.git"
REPO = "/content/cutoff-flood"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=False)   # get the latest version
sys.path.insert(0, REPO)
from src import config
print("Project code loaded from", REPO)

Project code loaded from /content/cutoff-flood


In [3]:
import datetime, math, time
import osmnx as ox
import pandas as pd
import geopandas as gpd

TODAY = datetime.date.today().isoformat()
RAW = f"{REPO}/data/raw"
WORK = "/content/osm_work"
RESULTS = f"{REPO}/results"
for d in (RAW, WORK, RESULTS):
    os.makedirs(d, exist_ok=True)

# To test the backup area instead, use: DISTRICTS = config.BACKUP_DISTRICT
DISTRICTS = config.DISTRICTS

# 1. Download the OSM file for northeast India once (dated copy, for reproducibility)
PBF_URL = "https://download.geofabrik.de/asia/india/north-eastern-zone-latest.osm.pbf"
PBF = f"{RAW}/osm_northeast_{TODAY}.osm.pbf"
if not os.path.exists(PBF):
    subprocess.run(["wget", "-q", "-O", PBF, PBF_URL], check=True)
print("OSM file:", PBF, f"({os.path.getsize(PBF) / 1e6:.0f} MB)")


def osmium(*args):
    """Run an osmium command and stop with a clear message if it fails."""
    r = subprocess.run(["osmium", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr)


def get_boundary(place, tries=4):
    """District boundary from Nominatim (a different, usually reliable server)."""
    wait = 10
    for attempt in range(1, tries + 1):
        try:
            return ox.geocode_to_gdf(place)
        except Exception as e:
            if attempt == tries:
                raise
            print(f"  boundary lookup failed ({str(e)[:60]}), retrying in {wait}s")
            time.sleep(wait); wait *= 2


def road_class(h):
    """OSM 'highway' can be a list; keep the first value."""
    return h[0] if isinstance(h, list) else h


NOT_ROADS = {"proposed", "construction", "abandoned", "platform", "raceway", "bus_stop", "elevator", "corridor"}
HEALTH_TAGS = {"amenity": ["hospital", "clinic", "doctors"], "healthcare": True}

OSM file: /content/cutoff-flood/data/raw/osm_northeast_2026-10-26.osm.pbf (110 MB)


In [4]:
def roads_and_health(area_pbf, tag):
    """From a cut-out .pbf, build the road graph and the health facility table."""
    roads_osm = f"{WORK}/roads_{tag}.osm"
    health_osm = f"{WORK}/health_{tag}.osm"
    osmium("tags-filter", area_pbf, "w/highway", "-o", roads_osm, "-O")
    osmium("tags-filter", area_pbf, "nwr/amenity=hospital,clinic,doctors", "nwr/healthcare", "-o", health_osm, "-O")
    # simplify=False keeps every road segment with its own type (merging can mix types)
    G = ox.graph_from_xml(roads_osm, simplify=False, retain_all=True)
    try:
        health = ox.features_from_xml(health_osm, tags=HEALTH_TAGS)
    except Exception:
        health = gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    return G, health


def check_district(name, info):
    print(f"\n=== {name} ===")
    try:
        boundary = get_boundary(info["place"])
        poly = f"{WORK}/{name}_boundary.geojson"
        boundary[["geometry"]].to_file(poly, driver="GeoJSON")
        area_pbf = f"{WORK}/{name}.osm.pbf"
        osmium("extract", "-p", poly, PBF, "-o", area_pbf, "-O")
        how = "district boundary"
        area_km2 = boundary.to_crs(boundary.estimate_utm_crs()).area.sum() / 1e6
    except Exception as e:
        print("WARNING: boundary lookup failed; using the rough box. "
              "These numbers overlap other districts. Rerun later.", str(e)[:100])
        l, b, r, t = info["bbox"]
        area_pbf = f"{WORK}/{name}_box.osm.pbf"
        osmium("extract", "-b", f"{l},{b},{r},{t}", PBF, "-o", area_pbf, "-O")
        how = "rough box (NOT comparable, rerun)"
        area_km2 = float("nan")

    G, health = roads_and_health(area_pbf, name)
    ox.save_graphml(G, f"{RAW}/roads_{name}_{TODAY}.graphml")
    if len(health):
        keep = ["geometry"] + [c for c in ["name", "amenity", "healthcare"] if c in health.columns]
        health.reset_index()[keep].to_file(f"{RAW}/health_{name}_{TODAY}.geojson", driver="GeoJSON")

    # A two-way road is stored twice (once per direction). Count each road once.
    edges = ox.graph_to_gdfs(ox.convert.to_undirected(G), nodes=False)
    edges["cls"] = edges["highway"].apply(road_class)
    edges = edges[~edges["cls"].isin(NOT_ROADS)]
    edges["km"] = edges.to_crs(edges.estimate_utm_crs()).length / 1000

    km_by_class = edges.groupby("cls")["km"].sum().sort_values(ascending=False)
    total_km = km_by_class.sum()
    minor_km = km_by_class[km_by_class.index.isin(config.MINOR_ROAD_TYPES)].sum()
    major_km = km_by_class[km_by_class.index.isin(config.MAJOR_ROAD_TYPES)].sum()

    print(f"Area used: {how}")
    print("Road length by type (km), top 12:")
    print(km_by_class.head(12).round(1).to_string())

    return {
        "district": name,
        "area_used": how,
        "area_km2": round(area_km2, 0),
        "road_km_total": round(total_km, 0),
        "road_density_km_per_km2": round(total_km / area_km2, 2) if area_km2 == area_km2 else None,
        "minor_road_share": round(minor_km / total_km, 2) if total_km else 0,
        "major_road_share": round(major_km / total_km, 2) if total_km else 0,
        "health_facilities": len(health),
    }


rows = [check_district(n, i) for n, i in DISTRICTS.items()]
summary = pd.DataFrame(rows)
summary.to_csv(f"{RESULTS}/road_check_{TODAY}.csv", index=False)
summary


=== Sivasagar ===
Area used: district boundary
Road length by type (km), top 12:
cls
residential      613.9
unclassified     442.1
tertiary         377.8
primary          123.3
trunk            117.7
path             107.1
secondary         53.2
track             47.8
living_street     12.1
service            8.5
footway            0.8
trunk_link         0.0

=== Charaideo ===
Area used: district boundary
Road length by type (km), top 12:
cls
residential      551.9
unclassified     397.9
tertiary         278.8
path             159.1
primary          126.5
trunk            106.1
track             86.1
secondary         54.0
service           22.8
living_street     13.3

=== Jorhat ===
Area used: district boundary
Road length by type (km), top 12:
cls
residential      867.1
unclassified     645.6
tertiary         338.0
primary          164.9
trunk            131.7
track            104.9
service           97.2
secondary         51.8
road               9.7
path               6.2
tertiary_

,district,area_used,area_km2,road_km_total,road_density_km_per_km2,minor_road_share,major_road_share,health_facilities
0,Sivasagar,district boundary,1574.0,1904.0,1.21,0.59,0.35,37
1,Charaideo,district boundary,1067.0,1797.0,1.68,0.60,0.31,27
2,Jorhat,district boundary,2051.0,2428.0,1.18,0.71,0.28,80


## How to read the numbers

These are rough guides, not strict rules:

| Measure | Looks good | Warning sign |
|---|---|---|
| `minor_road_share` | above 0.6 (most road length is village roads) | below 0.4 (mostly highways) |
| `road_density_km_per_km2` | above about 1 | well below 0.5 |
| `health_facilities` | 10 or more per district | fewer than 5 |

`area_used` must say **district boundary** for every row. The numbers alone are not enough: do the visual check below.

## Visual check: OSM roads on top of satellite imagery

Yellow = village and minor roads in OSM. Red = main roads. Blue markers = health facilities.

**Important:** roads are only drawn within `RADIUS_M` (3 km) of `CENTER`. Blank areas beyond that square are expected, not missing data.

**Towns are almost always mapped well, so do not test them.** Set `CENTER` to rural villages 10 to 20 km from the towns, where people actually get cut off. Good choices are areas reported as hit in July 2026: Nazira, Gaurisagar, Amguri, Demow (Sivasagar) and villages in Charaideo. Right-click a village in Google Maps to copy its coordinates.

At each village, look at the satellite image: small roads and tracks with **no yellow line on top** are missing from OSM. Record how many of 4 villages look complete in `NOTES.md`.

In [5]:
import folium

# Paste a RURAL village (lat, lon) from Google Maps here, not a town.
CENTER = DISTRICTS["Sivasagar"]["town"]   # change this
RADIUS_M = 3000

lat, lon = CENTER
dlat = RADIUS_M / 111_000
dlon = RADIUS_M / (111_000 * math.cos(math.radians(lat)))
box_pbf = f"{WORK}/view.osm.pbf"
osmium("extract", "-b", f"{lon - dlon},{lat - dlat},{lon + dlon},{lat + dlat}", PBF, "-o", box_pbf, "-O")
Gv, hv = roads_and_health(box_pbf, "view")

e = ox.graph_to_gdfs(ox.convert.to_undirected(Gv), nodes=False).reset_index()
e["cls"] = e["highway"].apply(road_class)
e = e[~e["cls"].isin(NOT_ROADS)][["cls", "geometry"]]

m = folium.Map(location=CENTER, zoom_start=15, tiles=None)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery", name="Satellite").add_to(m)
folium.GeoJson(e[e["cls"].isin(config.MAJOR_ROAD_TYPES)],
               style_function=lambda x: {"color": "red", "weight": 3}, name="Main roads").add_to(m)
folium.GeoJson(e[~e["cls"].isin(config.MAJOR_ROAD_TYPES)],
               style_function=lambda x: {"color": "yellow", "weight": 2}, name="Minor roads").add_to(m)
for _, r in hv.iterrows():
    p = r.geometry.centroid
    folium.Marker([p.y, p.x], popup=str(r.get("name", "health facility"))).add_to(m)
folium.LayerControl().add_to(m)
m

## Decision

Write the results in `NOTES.md` (the table under "Check 1"). Use *File > Save a copy in Drive* to keep this run; do **not** save it back to GitHub.

- **Pass:** village roads are mostly present in at least one district. Keep that district (or all three).
- **Partial:** good in one district only. Shrink the study area to that district.
- **Fail:** village roads mostly missing everywhere. Set `DISTRICTS = config.BACKUP_DISTRICT` at the top and run again for Barpeta.